# 070 — Fusión multimodal y representación conjunta

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Tres estrategias de fusión:** **temprana** (concatenar señales/características y entrenar
un modelo único: interacciones ricas, frágil ante modalidades faltantes), **tardía** (un
modelo por modalidad y combinar decisiones — promedio, producto, votación: modular y
robusta, pero ciega a interacciones) e **intermedia** (codificadores separados que se
comunican dentro de la red, típicamente con **atención cruzada** — el estándar actual).

**Atención cruzada:** Q sale de la modalidad A; K y V de la B:
`Atención(Q,K,V) = softmax(Q·Kᵀ/√d)·V`. Cada elemento de A pregunta qué partes de B le son
relevantes y recibe un resumen ponderado. La dirección importa (texto→imagen ≠
imagen→texto). Flamingo inserta estas capas en un LLM congelado.

**Alineación de espacios:** contrastiva (CLIP), proyección aprendida al espacio de tokens
del LLM (LLaVA), CCA como precursor clásico. También hay que alinear tiempo y granularidad.

**Problemas prácticos:** modalidades faltantes (la tardía degrada con gracia; la temprana
necesita *dropout de modalidades*) y **dominancia** (una modalidad predictiva hace que el
modelo ignore la otra — se diagnostica con ablaciones).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Puntajes: `Q·K₁/√2 = 0` y `Q·K₂/√2 = 2/1.414 = 1.414`. Softmax:
`exp(0)=1.00`, `exp(1.414)=4.11` → pesos `(0.196, 0.804)`. Salida:
`0.196·(1,0) + 0.804·(0,1) = (0.196, 0.804)`. Gana el parche 2 porque su clave apunta en la
dirección de la consulta: la atención mide compatibilidad Q-K, no importancia absoluta.

**Ejercicio 2.** Promedio: `(0.31, 0.69)` → clase 2. Producto: `(0.6·0.02, 0.4·0.98) =
(0.012, 0.392)` → renormalizado `(0.030, 0.970)` → clase 2 con mucho más margen. El
producto castiga más el desacuerdo: equivale a sumar log-probabilidades, y una modalidad
que asigna ~0 a una clase la **veta** casi por completo aunque la otra la apoye. El
promedio nunca veta: conserva toda clase que alguna modalidad apoye.

**Ejercicio 3.** (a) La temprana recibe una entrada que nunca vio en entrenamiento (mitad
del vector en cero): el modelo opera fuera de distribución y su salida es imprevisible, no
simplemente "menos precisa". (b) La tardía pierde un voto pero el clasificador de audio
sigue funcionando: degradación controlada. El *dropout de modalidades* (anular al azar una
modalidad durante el entrenamiento) enseña a la temprana a operar con entradas ausentes.

**Ejercicio 4.** Implementación debajo: reproduce pesos (0.196, 0.804) y salida
(0.196, 0.804).


In [ ]:
result = run_lab("attention", seed=70)
assert result["kind"] == "attention"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — verificado con código
import math

def cross_attention(Q, Ks, Vs, d=2):
    scores = [sum(q * k for q, k in zip(Q, K)) / math.sqrt(d) for K in Ks]
    exps = [math.exp(s) for s in scores]
    total = sum(exps)
    pesos = [e / total for e in exps]
    dim = len(Vs[0])
    salida = [sum(p * V[i] for p, V in zip(pesos, Vs)) for i in range(dim)]
    return [round(p, 3) for p in pesos], [round(x, 3) for x in salida]

pesos, salida = cross_attention((0, 2), [(1, 0), (0, 1)], [(1, 0), (0, 1)])
print("pesos:", pesos, "| salida:", salida)


In [ ]:
# Ejercicio 2 — promedio vs producto en fusión tardía
audio = (0.6, 0.4)
vision = (0.02, 0.98)
promedio = [(a + v) / 2 for a, v in zip(audio, vision)]
producto = [a * v for a, v in zip(audio, vision)]
total = sum(producto)
producto_norm = [round(p / total, 3) for p in producto]
print("promedio:", promedio, "| producto renormalizado:", producto_norm)


## Reflexión

1. Tu modelo audiovisual rinde casi igual cuando anulas por completo el audio. ¿Qué
   fenómeno es, con qué experimento lo confirmas y qué cambiarías en el entrenamiento?
2. ¿Por qué el sarcasmo (texto positivo + tono de voz negativo) es indetectable para una
   fusión tardía **por diseño**, y cuál es la modificación mínima que lo haría detectable?
3. En Flamingo la atención cruzada usa Q del texto y K,V de la imagen. ¿Qué cambiaría
   conceptualmente si fuera al revés, y por qué esa dirección es la adecuada cuando el
   objetivo es generar texto?
